# PT-W3-D1 概念实验：Agent 如何利用 Ontology

把 Ontology 当作 Agent 的企业语义地图：实体识别、关系导航、状态推理、规则解释和能力建议。实验对比声明表命中与无声明时的 fallback。

## 实验 1：声明式查询与语义导航

核心概念来自 D1：Ontology 提供 Concept 词汇、Relationship、Lifecycle、Rule、Capability，而不是一堆孤立数据表。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc'
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False

ONTOLOGY = {
    'Space': {'aliases': {'铺位', '商铺', '位置'}, 'lifecycle': ['空置', '锁定', '已租', '退出'], 'rule': '未完成退租流程的 Space 不可出租', 'capability': '查询铺位可用性'},
    'Lease': {'aliases': {'租约', '合同'}, 'lifecycle': ['草案', '生效', '终止中', '已终止'], 'rule': 'Inspection 完成后才释放 Space', 'capability': '查询合同摘要'},
    'Tenant': {'aliases': {'租户', '商户'}, 'lifecycle': ['潜在', '签约', '履约中'], 'rule': '商户身份由 Merchant Context 唯一拥有', 'capability': '查询商户档案'},
}
RELATIONS = {('Lease', 'Space'): ('occupies', '租约占用铺位'), ('Tenant', 'Lease'): ('signs', '商户签署租约')}
print('Ontology 实体:', list(ONTOLOGY))
print('语义关系:', RELATIONS)

In [ ]:
def ontology_lookup(term):
    for entity, spec in ONTOLOGY.items():
        if term == entity or term in spec['aliases']:
            return {'kind': 'declaration_hit', 'entity': entity, 'meaning': spec}
    return {'kind': 'fallback', 'entity': None, 'meaning': {'message': f'未找到声明：{term}，只能返回原始关键词，不能安全推理'}}

def answer(question):
    terms = ['铺位' if 'A101' in question else '', '租约' if '合同' in question else '']
    hits = [ontology_lookup(t) for t in terms if t]
    if not hits or any(h['kind'] == 'fallback' for h in hits):
        return 'fallback：缺少 Ontology 声明，建议转人工或补充语义定义。'
    space, lease = hits[0], hits[-1]
    return (f"declaration_hit：A101 -> {space['entity']}；{lease['entity']} --occupies--> Space；"
            f"生命周期含 {lease['meaning']['lifecycle']}；规则：{lease['meaning']['rule']}；"
            f"建议能力：{space['meaning']['capability']}")

for q in ['A101 铺位的合同为什么不能出租？', 'A101 的未知状态是什么？']:
    print(q); print(answer(q))

In [ ]:
queries = ['铺位', '租约', '商户', '发票']
results = [ontology_lookup(q) for q in queries]
hit_count = sum(r['kind'] == 'declaration_hit' for r in results)
fallback_count = len(results) - hit_count
print(f'声明命中={hit_count}，fallback={fallback_count}')
for q, r in zip(queries, results): print(f'{q}: {r["kind"]}')

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['声明表命中', 'fallback'], [hit_count, fallback_count], color=['#2a9d8f', '#e76f51'])
ax.set_title('Agent 查询路径'); ax.set_ylabel('查询数'); ax.set_ylim(0, len(queries) + 1)
for i, v in enumerate([hit_count, fallback_count]): ax.text(i, v + .05, str(v), ha='center')
plt.tight_layout(); plt.show()